In [ ]:
import numpy as np
import pandas as pd

lob = pd.read_csv("lob.csv")
trades = pd.read_csv("trades.csv")

lob["bid"] = lob["bids[0].price"]
lob["ask"] = lob["asks[0].price"]

lob["bid_vol"] = lob["bids[0].amount"]
lob["ask_vol"] = lob["asks[0].amount"]

lob["mid"] = (lob["bid"] + lob["ask"]) / 2
lob["spread"] = lob["ask"] - lob["bid"]

tick_size = lob["spread"].replace(0, np.nan).median()

lob["microprice"] = (
    lob["ask"] * lob["bid_vol"] +
    lob["bid"] * lob["ask_vol"]
) / (lob["bid_vol"] + lob["ask_vol"] + 1e-9)

lob["imbalance_1"] = (
    lob["bid_vol"] - lob["ask_vol"]
) / (lob["bid_vol"] + lob["ask_vol"] + 1e-9)

lob["mid_ret"] = lob["mid"].pct_change()
lob["vol"] = lob["mid_ret"].rolling(200).std().fillna(0)

print("LOB shape:", lob.shape)
print("Trades shape:", trades.shape)
print("Tick size:", tick_size)

display(lob[[
    "local_timestamp", "bid", "ask", "mid",
    "spread", "microprice", "imbalance_1", "vol"
]].head())

display(trades.head())

In [ ]:
lob[["mid", "spread", "microprice", "imbalance_1", "vol"]].describe()

In [ ]:

levels_list = [5, 10, 20, 25]

for L in levels_list:
    
    bid_cols = [f"bids[{i}].amount" for i in range(L)]
    ask_cols = [f"asks[{i}].amount" for i in range(L)]

    bid_sum = lob[bid_cols].sum(axis=1)
    ask_sum = lob[ask_cols].sum(axis=1)

    lob[f"imbalance_{L}"] = (
        bid_sum - ask_sum
    ) / (bid_sum + ask_sum + 1e-9)

    lob[f"bid_ask_ratio_{L}"] = (
        bid_sum + 1e-9
    ) / (ask_sum + 1e-9)

    lob[f"imbalance_{L}_avg"] = (
        lob[f"imbalance_{L}"]
        .ewm(span=200, adjust=False)
        .mean()
    )

    lob[f"bid_ask_ratio_{L}_avg"] = (
        lob[f"bid_ask_ratio_{L}"]
        .ewm(span=200, adjust=False)
        .mean()
    )

feature_cols = []

for L in levels_list:
    feature_cols += [
        f"imbalance_{L}",
        f"bid_ask_ratio_{L}",
        f"imbalance_{L}_avg",
        f"bid_ask_ratio_{L}_avg",
    ]

print("Number of features:", len(feature_cols))

display(lob[feature_cols].head())

In [ ]:
trades["signed_amount"] = np.where(
    trades["side"] == "buy",
    trades["amount"],
    -trades["amount"]
)

trades["notional"] = trades["price"] * trades["amount"]

trades["signed_notional"] = np.where(
    trades["side"] == "buy",
    trades["notional"],
    -trades["notional"]
)

lob_times = lob["local_timestamp"].values

idx = np.searchsorted(
    lob_times,
    trades["local_timestamp"].values,
    side="right"
) - 1

valid = (idx >= 0) & (idx < len(lob))

trades_tmp = trades.loc[valid].copy()
trades_tmp["lob_idx"] = idx[valid]

trade_features = trades_tmp.groupby("lob_idx").agg(
    trade_count=("amount", "count"),
    total_volume=("amount", "sum"),
    signed_volume=("signed_amount", "sum"),
    total_notional=("notional", "sum"),
    signed_notional=("signed_notional", "sum"),
).reset_index()

lob["lob_idx"] = lob.index

lob = lob.merge(
    trade_features,
    on="lob_idx",
    how="left"
)

trade_cols = [
    "trade_count",
    "total_volume",
    "signed_volume",
    "total_notional",
    "signed_notional"
]

for c in trade_cols:
    lob[c] = lob[c].fillna(0)


lob["trade_imbalance"] = (
    lob["signed_volume"]
    / (lob["total_volume"] + 1e-9)
)

lob["notional_imbalance"] = (
    lob["signed_notional"]
    / (lob["total_notional"] + 1e-9)
)

lob["trade_imbalance_avg"] = (
    lob["trade_imbalance"]
    .ewm(span=100, adjust=False)
    .mean()
)

lob["notional_imbalance_avg"] = (
    lob["notional_imbalance"]
    .ewm(span=100, adjust=False)
    .mean()
)

display(lob[[
    "trade_count",
    "trade_imbalance",
    "trade_imbalance_avg",
    "notional_imbalance",
    "notional_imbalance_avg"
]].head())

In [ ]:
signal_cols = [
    "trade_imbalance",
    "trade_imbalance_avg",
    "notional_imbalance",
    "notional_imbalance_avg",

    "imbalance_1",
    "imbalance_5",
    "imbalance_10",
    "imbalance_20",
    "imbalance_25",

    "imbalance_5_avg",
    "imbalance_10_avg",
    "imbalance_20_avg",
    "imbalance_25_avg",
]

for c in signal_cols:
    lob[f"{c}_lag1"] = lob[c].shift(1)

horizon = 1

lob["future_mid_return"] = (
    lob["mid"].shift(-horizon) - lob["mid"]
) / lob["mid"]


corr_df_lagged = pd.DataFrame({
    "feature": [f"{c}_lag1" for c in signal_cols],
    "corr_future_return": [
        lob[f"{c}_lag1"].corr(lob["future_mid_return"])
        for c in signal_cols
    ]
})

corr_df_lagged["abs_corr"] = (
    corr_df_lagged["corr_future_return"].abs()
)

corr_df_lagged = corr_df_lagged.sort_values(
    "abs_corr",
    ascending=False
).reset_index(drop=True)

corr_df_lagged

In [ ]:
lob["alpha_signal"] = (
    0.35 * lob["imbalance_5_lag1"] +
    0.25 * lob["imbalance_10_lag1"] +
    0.20 * lob["imbalance_1_lag1"] +
    0.20 * lob["trade_imbalance_lag1"]
)

lob["alpha_signal"] = np.tanh(
    lob["alpha_signal"]
)

print(
    "Signal correlation:",
    lob["alpha_signal"].corr(
        lob["future_mid_return"]
    )
)

lob[[
    "alpha_signal",
    "future_mid_return"
]].head()

In [ ]:
class Backtester:

    def __init__(self, lob):

        self.lob = lob

        self.inventory = 0.0
        self.cash = 0.0

        self.bid_order = None
        self.ask_order = None

        self.trades = []

        self.inventory_curve = []
        self.cash_curve = []
        self.equity_curve = []

    def place_orders(
        self,
        bid_price,
        ask_price,
        size=1.0
    ):
        self.bid_order = (bid_price, size)
        self.ask_order = (ask_price, size)

    def cancel_orders(self):

        self.bid_order = None
        self.ask_order = None

    def process_fills(
        self,
        next_bid,
        next_ask,
        t
    ):

        if self.bid_order is not None:

            price, size = self.bid_order

            if next_ask <= price:

                self.inventory += size
                self.cash -= price * size

                self.trades.append(
                    ("buy", price, size, t)
                )

                self.bid_order = None

        if self.ask_order is not None:

            price, size = self.ask_order

            if next_bid >= price:

                self.inventory -= size
                self.cash += price * size

                self.trades.append(
                    ("sell", price, size, t)
                )

                self.ask_order = None

    def mark_to_market(
        self,
        mid_price
    ):

        equity = (
            self.cash
            + self.inventory * mid_price
        )

        self.inventory_curve.append(
            self.inventory
        )

        self.cash_curve.append(
            self.cash
        )

        self.equity_curve.append(
            equity
        )

    def run(
        self,
        strategy
    ):

        for t in range(len(self.lob) - 1):

            row = self.lob.iloc[t]
            next_row = self.lob.iloc[t + 1]

            bid_quote, ask_quote = strategy(
                row,
                self.inventory
            )

            self.cancel_orders()

            self.place_orders(
                bid_quote,
                ask_quote
            )

            self.process_fills(
                next_row["bid"],
                next_row["ask"],
                t + 1
            )

            self.mark_to_market(
                next_row["mid"]
            )

        final_mid = self.lob.iloc[-1]["mid"]

        pnl = (
            self.cash
            + self.inventory * final_mid
        )

        turnover = sum(
            abs(price * size)
            for _, price, size, _
            in self.trades
        )

        return {

            "pnl": pnl,

            "inventory": self.inventory,

            "turnover": turnover,

            "trades": self.trades,

            "inventory_curve": self.inventory_curve,

            "cash_curve": self.cash_curve,

            "equity_curve": self.equity_curve,
        }

In [ ]:
def dummy_strategy(row, inventory):
    bid_quote = row["bid"]
    ask_quote = row["ask"]
    return bid_quote, ask_quote


bt = Backtester(lob)
res_dummy = bt.run(dummy_strategy)

print("Dummy strategy")
print("PnL:", res_dummy["pnl"])
print("Final inventory:", res_dummy["inventory"])
print("Turnover:", res_dummy["turnover"])
print("Trades:", len(res_dummy["trades"]))

In [ ]:
def calc_metrics(result, name):
    trades = result["trades"]
    equity = np.array(result["equity_curve"], dtype=float)
    inv = np.array(result["inventory_curve"], dtype=float)

    pnl = result["pnl"]
    turnover = result["turnover"]

    buy_trades = sum(1 for side, price, size, t in trades if side == "buy")
    sell_trades = sum(1 for side, price, size, t in trades if side == "sell")

    if len(equity) > 0:
        running_max = np.maximum.accumulate(equity)
        drawdown = equity - running_max
        max_drawdown = drawdown.min()
        equity_std = equity.std()
    else:
        max_drawdown = 0.0
        equity_std = 0.0

    if len(inv) > 0:
        mean_inventory = inv.mean()
        max_inventory_seen = inv.max()
        min_inventory_seen = inv.min()
        inventory_std = inv.std()
    else:
        mean_inventory = 0.0
        max_inventory_seen = 0.0
        min_inventory_seen = 0.0
        inventory_std = 0.0

    return {
        "strategy": name,
        "pnl": pnl,
        "final_inventory": result["inventory"],
        "turnover": turnover,
        "trades": len(trades),
        "buy_trades": buy_trades,
        "sell_trades": sell_trades,
        "pnl_per_trade": pnl / len(trades) if len(trades) > 0 else 0.0,
        "pnl_per_turnover": pnl / turnover if turnover > 0 else 0.0,
        "mean_inventory": mean_inventory,
        "max_inventory": max_inventory_seen,
        "min_inventory": min_inventory_seen,
        "inventory_std": inventory_std,
        "max_drawdown": max_drawdown,
        "equity_std": equity_std,
    }


metrics_dummy = calc_metrics(res_dummy, "Dummy best bid/ask")

pd.DataFrame([metrics_dummy])

In [ ]:
def avellaneda_stoikov_strategy(
    row,
    inventory,

    gamma=0.05,
    inv_penalty=0.01,

    base_spread_ticks=2.0,

    max_inventory=100
):


    ref_price = row["mid"]


    sigma = row["vol"]

    if np.isnan(sigma) or sigma <= 0:
        sigma = 1e-6

    tick = row["spread"]

    if tick <= 0:
        tick = 1e-7

    reservation_price = (
        ref_price
        - inventory * inv_penalty * tick
    )

    half_spread = (
        base_spread_ticks * tick
        + gamma * sigma * ref_price
    )

    bid_quote = (
        reservation_price
        - half_spread
    )

    ask_quote = (
        reservation_price
        + half_spread
    )

    if inventory >= max_inventory:
        bid_quote = -np.inf

    if inventory <= -max_inventory:
        ask_quote = np.inf

    return bid_quote, ask_quote


bt = Backtester(lob)

res_as = bt.run(
    avellaneda_stoikov_strategy
)

print("Avellaneda-Stoikov (2008)")
print("PnL:", res_as["pnl"])
print("Final inventory:", res_as["inventory"])
print("Turnover:", res_as["turnover"])
print("Trades:", len(res_as["trades"]))

In [ ]:
metrics_as = calc_metrics(
    res_as,
    "Avellaneda-Stoikov 2008"
)

pd.DataFrame([
    metrics_dummy,
    metrics_as
])

In [ ]:
def enhanced_as_strategy(
    row,
    inventory,

    gamma=0.05,

    inv_penalty=0.01,

    signal_weight=2.0,

    base_spread_ticks=2.5,

    max_inventory=100
):

    ref_price = row["microprice"]

    sigma = row["vol"]

    if np.isnan(sigma) or sigma <= 0:
        sigma = 1e-6

    tick = row["spread"]

    if tick <= 0:
        tick = 1e-7

    alpha_signal = row["alpha_signal"]

    if np.isnan(alpha_signal):
        alpha_signal = 0.0

    reservation_price = (
        ref_price

        - inventory * inv_penalty * tick

        + signal_weight * alpha_signal * tick
    )

    half_spread = (
        base_spread_ticks * tick
        + gamma * sigma * ref_price
    )

    bid_quote = (
        reservation_price
        - half_spread
    )

    ask_quote = (
        reservation_price
        + half_spread
    )
    if inventory >= max_inventory:
        bid_quote = -np.inf

    if inventory <= -max_inventory:
        ask_quote = np.inf

    return bid_quote, ask_quote

bt = Backtester(lob)

res_enhanced = bt.run(
    enhanced_as_strategy
)

print("Enhanced Avellaneda-Stoikov")
print("PnL:", res_enhanced["pnl"])
print("Final inventory:", res_enhanced["inventory"])
print("Turnover:", res_enhanced["turnover"])
print("Trades:", len(res_enhanced["trades"]))

In [ ]:
metrics_enhanced = calc_metrics(
    res_enhanced,
    "Enhanced AS"
)

comparison_df = pd.DataFrame([
    metrics_dummy,
    metrics_as,
    metrics_enhanced
])

comparison_df

In [ ]:
import itertools

results = []

signal_weights = [0.5, 1.0, 2.0, 3.0]
spreads = [1.5, 2.0, 2.5, 3.0]
inv_penalties = [0.005, 0.01, 0.02]

runs = list(itertools.product(
    signal_weights,
    spreads,
    inv_penalties
))

print("Total runs:", len(runs))

for signal_weight, spread, inv_penalty in runs:

    def strategy(row, inventory):

        ref_price = row["microprice"]

        sigma = row["vol"]

        if np.isnan(sigma) or sigma <= 0:
            sigma = 1e-6

        tick = row["spread"]

        if tick <= 0:
            tick = 1e-7

        alpha_signal = row["alpha_signal"]

        if np.isnan(alpha_signal):
            alpha_signal = 0.0

        reservation_price = (
            ref_price
            - inventory * inv_penalty * tick
            + signal_weight * alpha_signal * tick
        )

        half_spread = (
            spread * tick
            + 0.05 * sigma * ref_price
        )

        bid_quote = (
            reservation_price
            - half_spread
        )

        ask_quote = (
            reservation_price
            + half_spread
        )

        if inventory >= 100:
            bid_quote = -np.inf

        if inventory <= -100:
            ask_quote = np.inf

        return bid_quote, ask_quote

    bt = Backtester(lob)

    res_tmp = bt.run(strategy)

    metrics_tmp = calc_metrics(
        res_tmp,
        f"sw={signal_weight}_sp={spread}_inv={inv_penalty}"
    )

    metrics_tmp["signal_weight"] = signal_weight
    metrics_tmp["spread"] = spread
    metrics_tmp["inv_penalty"] = inv_penalty

    results.append(metrics_tmp)

    print(
        signal_weight,
        spread,
        inv_penalty,
        "PnL:",
        round(metrics_tmp["pnl"], 6)
    )


sweep_df = pd.DataFrame(results)

sweep_df = sweep_df.sort_values(
    "pnl",
    ascending=False
).reset_index(drop=True)

sweep_df.head(20)

In [ ]:
def best_strategy(
    row,
    inventory
):

    signal_weight = 1.0
    spread = 3.0
    inv_penalty = 0.005

    ref_price = row["microprice"]

    sigma = row["vol"]

    if np.isnan(sigma) or sigma <= 0:
        sigma = 1e-6

    tick = row["spread"]

    if tick <= 0:
        tick = 1e-7

    alpha_signal = row["alpha_signal"]

    if np.isnan(alpha_signal):
        alpha_signal = 0.0

    reservation_price = (
        ref_price
        - inventory * inv_penalty * tick
        + signal_weight * alpha_signal * tick
    )

    half_spread = (
        spread * tick
        + 0.05 * sigma * ref_price
    )

    bid_quote = (
        reservation_price
        - half_spread
    )

    ask_quote = (
        reservation_price
        + half_spread
    )

    if inventory >= 100:
        bid_quote = -np.inf

    if inventory <= -100:
        ask_quote = np.inf

    return bid_quote, ask_quote

bt = Backtester(lob)

res_best = bt.run(best_strategy)

metrics_best = calc_metrics(
    res_best,
    "Best Enhanced AS"
)

pd.DataFrame([metrics_best])

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(
    res_dummy["equity_curve"],
    label="Dummy"
)

plt.plot(
    res_as["equity_curve"],
    label="AS 2008"
)

plt.plot(
    res_best["equity_curve"],
    label="Best Enhanced AS"
)

plt.title("Equity Curves")

plt.xlabel("Time")
plt.ylabel("PnL")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(
    res_dummy["inventory_curve"],
    label="Dummy"
)

plt.plot(
    res_as["inventory_curve"],
    label="AS 2008"
)

plt.plot(
    res_best["inventory_curve"],
    label="Best Enhanced AS"
)

plt.title("Inventory Curves")

plt.xlabel("Time")
plt.ylabel("Inventory")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
final_summary = pd.DataFrame([

    calc_metrics(
        res_dummy,
        "Dummy best bid/ask"
    ),

    calc_metrics(
        res_as,
        "Avellaneda-Stoikov 2008"
    ),

    calc_metrics(
        res_best,
        "Best Enhanced AS"
    )

])

final_summary = final_summary[[
    "strategy",

    "pnl",
    "turnover",
    "trades",

    "final_inventory",
    "inventory_std",

    "pnl_per_trade",
    "pnl_per_turnover",

    "max_drawdown",
    "equity_std"
]]

final_summary = final_summary.sort_values(
    "pnl",
    ascending=False
).reset_index(drop=True)

final_summary